In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
import torch
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
# 1. Convert Numpy arrays to PyTorch Tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [ ]:


# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train,y_train)
test_dataset = TensorDataset(X_test, y_test)

print(f"train_dataset created with {len(train_dataset)} samples.")
print(f"test_dataset created with {len(test_dataset)} samples.")

In [ ]:


batch_size = 32

# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"train_loader created with {len(train_loader)} batches.")
print(f"test_loader created with {len(test_loader)} batches.")

In [ ]:
# 4. Print shape of one batch
images, ages = next(iter(train_loader))
print(f"Shape of images batch: {images.shape}")
print(f"Shape of ages batch: {ages.shape}")

In [ ]:
import matplotlib.pyplot as plt

# 5. Display sample images
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    # Images are in (C, H, W) format, convert to (H, W, C) for displaying
    img = images[i].permute(1, 2, 0).cpu().numpy()
    age = ages[i].item()
    axes[i].imshow(img)
    axes[i].set_title(f'Age: {int(age)}')
    axes[i].axis('off')
plt.show()

In [ ]:
# Task 1: Write your model class here:
import torch.nn as nn

class AgePredictor(nn.Module):
    def __init__(self):
        super().__init__()
        # Input layer: 3 channels * 36 height * 36 width = 3888 features
        self.fc1 = nn.Linear(3 * 36 * 36, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 1) # Output layer for regression (1 output feature)

    def forward(self, x):
        # Flatten the image tensor (batch_size, 3, 36, 36) to (batch_size, 3*36*36)
        x = x.view(x.size(0), -1)
        x = nn.functional.relu(self.fc1(x))
        x = nn.functional.relu(self.fc2(x))
        x = nn.functional.relu(self.fc3(x))
        x = self.fc4(x)
        return x

print("AgePredictor class defined.")

In [ ]:
# Task 2: Write your training loop here:

def train_epoch(model, data_loader, loss_fn, optimizer, device):
    model.train()  # Set the model to training mode
    running_loss = 0.0

    for images, ages in data_loader:
        images = images.to(device)
        ages = ages.to(device)
        ages = ages.view(-1, 1) # Reshape ages to match output of the model

        # Forward pass
        outputs = model(images)
        loss = loss_fn(outputs, ages)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(data_loader)

print("train_epoch function defined.")

In [ ]:
# Task 3: Write your validation loop here:
def validate_epoch(model, data_loader, loss_fn, device):
    model.eval()  # Set the model to evaluation mode
    running_loss = 0.0

    with torch.no_grad():  # Disable gradient calculation during validation
        for images, ages in data_loader:
            images = images.to(device)
            ages = ages.to(device)
            ages = ages.view(-1, 1) # Reshape ages to match output of the model

            # Forward pass
            outputs = model(images)
            loss = loss_fn(outputs, ages)

            running_loss += loss.item()

    # Return average loss for the epoch
    return running_loss / len(data_loader)

print("validate_epoch function defined.")

In [ ]:
import torch.optim as optim

# Task 4: Define device, model, loss, optimizer:

# 1. Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Instantiate the model and move it to the device
model = AgePredictor().to(device)
print("Model instantiated and moved to device.")

# 3. Define the loss function
loss_fn = nn.MSELoss()
print("Loss function (MSE) defined.")

# 4. Initialize the optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)
print("Optimizer (Adam) initialized.")

In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 20
train_losses = []
val_losses = []

print("Starting training...")

for epoch in range(num_epochs):
    # Train the model for one epoch
    train_loss = train_epoch(model,   train_loader, loss_fn, optimizer, device)
    train_losses.append(train_loss)

    # Validate the model for one epoch
    val_loss = validate_epoch(model, test_loader, loss_fn, device)
    val_losses.append(val_loss)

    print(f'Epoch {epoch+1}/{num_epochs}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

print("Training complete.")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

#  Create a new figure and an axes object for the plot.
plt.figure(figsize=(10,6))

#  Plot the train_losses list
plt.plot(train_losses,  label='Training Loss')

plt.plot(val_losses,label='Validation Loss')
plt.title('Training and Validation Loss Over Epochs')

#  Label the x-axis 'Epoch' and the y-axis 'Loss'
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

print("Training and Validation Loss plot displayed.")

In [ ]:
# Task 2 (Bonus): Write your code here:
model.eval() # Set the model to evaluation mode

dataiter = iter(test_loader)
images, actual_ages = next(dataiter)

# Move images to the correct device
images = images.to(device)
with torch.no_grad():
    predicted_ages = model(images)

# Convert tensors to numpy for plotting
images = images.cpu().permute(0, 2, 3, 1).numpy() # (N, C, H, W) to (N, H, W, C)
actual_ages = actual_ages.cpu().numpy()
predicted_ages = predicted_ages.cpu().numpy().flatten()

# Display sample images with predictions
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i in range(min(5, len(images))):
    img = images[i]
    actual_age = actual_ages[i]
    predicted_age = predicted_ages[i]

    axes[i].imshow(img)
    axes[i].set_title(f'Actual: {int(actual_age)}\nPred: {int(predicted_age)}', fontsize=10)
    axes[i].axis('off')
plt.suptitle('Sample Predictions vs. Actual Ages', fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print("Sample predictions plot displayed.")